In [ ]:
import autogen
import json
import os
from sentence_transformers import SentenceTransformer
import numpy as np
from collections import defaultdict
import statistics
import matplotlib.pyplot as plt
import copy
import random
import sys
from tqdm.auto import tqdm
import re

# Parameters (Needs to be configured to the specific task)

In [ ]:
def read_task_output_file(file_name):
    """
    Read the task output logs
    
    Args:
    - file_name (str): The single log file that wants to get evaluated.
    """
    f = open( file_name,"r").readlines()
    output_dictionary = ""
    for line in f:
        if "is_correct" not in line and "correct_ans" not in line and  "check_result"  not in line:
            output_dictionary += line
        elif "is_correct" in line:
            correctness = line.replace(",","").split(":")[-1].rstrip().strip()
    return [output_dictionary,correctness]

In [ ]:
task_prompt_instr = open('../symbolic_task_prompt.txt').read().strip()
success1 = open('../successful_symbolic_1.txt').read().strip()
success2 = open('../successful_symbolic_2.txt').read().strip()
failure1 = open('../unsuccessful_symbolic_1.txt').read().strip()
failure2 = open('../unsuccessful_symbolic_2.txt').read().strip()

In [ ]:
# prompt related
task = {
    "name": "Providing judgments and annotations related to social norms and their violations in Chinese conversations", 
    "description": task_prompt_instr,
    "examples": f'''
        Successful Example #1:
        {success1}

        Successful Example #2:
        {success2}

        Unsuccessful Example #1:
        {failure1}

        Unsuccessful Example #2:
        {failure2}
    '''.strip()
}

sys_msg = f"""Task: {task["name"]}.
Task description: {task["description"]}
{task['examples']}
"""

In [ ]:
# parameters, input path and output path
num_critic_seeds = 15
num_quantifier_seeds = 3
critic_agent_output_path_prefix = "outputs/norm_violation/criteria/task_based/norm_rel-autogen-"
final_summarized_criteria_output_path = "outputs/norm_violation/final_filtered_criteria.json"
# task_logs_input_path = "logs/norm_violation_logs/autogen"
quantifier_agent_one_at_a_time_output_path_prefix = "outputs/norm_violation/evaluated_problems-"
quantifier_agent_all_at_once_output_path_prefix = "outputs/norm_violation/15-seeds-all-in-one/evaluated_problems-"
quantifier_stdev_plot_output_path = "outputs/norm_violation/mean_stdev_comparison.png"
adversarial_examples_output_path = "outputs/norm_violation/adversarial_examples.json"
adversarial_testing_evaluated_results_output_path = "outputs/norm_violation/adversarial_testing_evaluated_results.json"
quantifier_adversarial_testing_plot_output_path = "outputs/norm_violation/adversarial_quantifier_performance.png"

In [ ]:
import pickle
test_data = pickle.load(open('../../self_verification_1.pkl', 'rb'))
tdata, gpt_data, resps, X, Y = test_data

In [ ]:
# load the original logs (directory might be structured differently)
original_test_cases = {}

idx = 0
for prompt, label in gpt_data:
    original_test_cases[idx] = ['relevant' if label else 'irrelevant', prompt]
    idx += 1

In [ ]:
# Which part of the task output to drop might be different
def create_adversarial_samples(text):
    original_sample = json.loads(text)
    adversarial_sample = copy.deepcopy(original_sample)
    adversarial_sample['messages'] = []
    
    for m in original_sample['messages']: 
        if m['role'] == 'assistant':
            original_content = m['content']
            original_content_list = list(filter(lambda x: len(x) > 0, original_content.replace('. ', '\n').split('\n')))
            frac = 0.5 
            inds = set(random.sample(list(range(len(original_content_list))), int(frac*len(original_content_list))))
            new_content_list = [n for i,n in enumerate(original_content_list) if i not in inds]
            new_content = '\n'.join(new_content_list)
            adversarial_sample['messages'].append({'content': new_content, 'role': 'assistant'})
        else:
            adversarial_sample['messages'].append(m)
    
    return json.dumps(adversarial_sample, indent=2)

In [ ]:
# LLM API
# config_list = autogen.config_list_from_json(
#     "OAI_CONFIG_LIST",
# )

%env AUTOGEN_USE_DOCKER=0
openai_key = '<put-your-key-here>' #Put your OpenAI key here
config_list = [{"model": "gpt-4o-mini", "api_key": openai_key}]

# Checkpoint 1: Run CriticAgent Multiple Times

In [ ]:
for i in range(num_critic_seeds):
        
    critic = autogen.AssistantAgent(
        name = "critic",
        llm_config = {"config_list": config_list,"cache_seed":i},
        system_message = """You are a helpful assistant. You suggest criteria for evaluating different tasks. They should be dinstinguishable, quantifieable and not redundant.
        Convert the evaluation criteria into a dictionary where the keys are the criteria.
        The value of each key is a dictionary as follows {"description": criteria description , "accepted_values": possible accepted inputs for this key}
        Make sure the keys are criteria for assessing the given task.  "accepted_values" include the acceptable inputs for each key that are fine-grained and preferrably mlti-graded levels. "description" includes the criterion description.
        Return the dictionary."""
    )

    critic_user = autogen.UserProxyAgent(
        name = "critic_user",
        max_consecutive_auto_reply = 0,  # terminate without auto-reply
        human_input_mode = "NEVER",
    )

    gen_criteria = critic_user.initiate_chat(critic, message=sys_msg)
    criteria = critic_user.last_message()
    with open(critic_agent_output_path_prefix + str(i) + ".json", "w") as cr_file:
        try:
            crit_content = re.findall(re.compile('{[\s\S]+}'), criteria["content"])[0]
            crit_dict = json.loads(crit_content)
            json.dump(crit_dict,cr_file,indent=2)
        except:
            print(sys.exc_info()[0])

# Checkpoint 2: Generate Summarized Criteria

In [ ]:
# load data
crit_dicts = []

for i in range(num_critic_seeds):
    with open(critic_agent_output_path_prefix + str(i) + ".json",'r') as fptr:
        try:
            criteria = json.load(fptr)
            if criteria:
                crit_dicts.append(criteria)
        except:
            print(sys.exc_info()[0])

In [ ]:
criteria_summarizer_message_base = """You are a helpful assistant. You suggest criteria for evaluating different tasks. They should be dinstinguishable, quantifieable and not redundant.
A criteria dictionary is a dictionary where the keys are the criteria. 
The value of each key is a dictionary as follows {"description": criteria description , "accepted_values": possible accepted inputs for this key}
You will be given a list of criteria dictionaries that others have suggested. They will be of varying qualities, and some of them will be synonymous.
You should pick the best 25 distinct criteria for the task, and each criterion's corresponding best decription and range of accepted values.
Your output should be a criteria dictionary containing the 25 distinct criteria you have picked.
Make sure the keys are criteria for assessing the given task.  "accepted_values" include the acceptable inputs for each key that are fine-grained and preferrably mlti-graded levels. "description" includes the criterion description.
Return only the dictionary, and in json format."""

criteria_summarizer = autogen.AssistantAgent(
    name = "criteria_summarizer",
    llm_config = {"config_list": config_list},
    system_message = criteria_summarizer_message_base,
)

In [ ]:
def get_summarized_criteria(crit_dicts):
    criteria_summarizer_user = autogen.UserProxyAgent(
        name = "criteria_summarizer_user",
        max_consecutive_auto_reply = 0,  # terminate without auto-reply
        human_input_mode = "NEVER",
    )    

    message = f"""Task: {task["name"]}.
Task description: {task["description"]}
Suggested criteria: {crit_dict}
"""

    criteria_summarizer_user.initiate_chat(criteria_summarizer, message=message)
    # return the last received from the criteria summarizer
    return criteria_summarizer_user.last_message()["content"]

In [ ]:
summarized_criteria_list = []

for i in range(len(crit_dicts) // 25):
    summarized_criteria_list.append(get_summarized_criteria(crit_dicts[i*25 : (i+1)*25]))

final_summarized_criteria = get_summarized_criteria(summarized_criteria_list)

In [ ]:
final_summarized_criteria = final_summarized_criteria[7:-3].strip()

In [ ]:
final_summarized_criteria = json.loads(final_summarized_criteria)

# convert to a list of words
conversion_dict_final = {}
for criteria in final_summarized_criteria:
    conversion_dict_final[' '.join(criteria.lower().split('_'))] = criteria

final_list_of_criteria = list(conversion_dict_final.keys())

In [ ]:
# set up synonymous detection
def find_synonymous_in_list(word_list, threshold=0.75):
    model = SentenceTransformer('paraphrase-MiniLM-L6-v2', device='cuda:1')  # Load a pre-trained SentenceTransformer model
    embeddings = model.encode(word_list, convert_to_tensor=True)  # Compute embeddings for the word list
    embeddings = embeddings.cpu()
    

    processed_words = set()
    used_in_synonyms = set()
    synonymous_dict = {}

    for i, word in enumerate(word_list):
        if word not in processed_words and word not in used_in_synonyms:
            synonymous_list = []
            for j, other_word in enumerate(word_list):
                if i != j:
                    cosine_sim = np.dot(embeddings[i], embeddings[j]) / (np.linalg.norm(embeddings[i]) * np.linalg.norm(embeddings[j]))
                    if cosine_sim > threshold:
                        synonymous_list.append(other_word)
                        used_in_synonyms.add(other_word)
            if synonymous_list or word not in synonymous_dict:  # Check if synonymous_list is not empty or word is not already added
                synonymous_list = list(set(synonymous_list))  # Remove duplicates from the list
                synonymous_list.sort()  # Sort the list to make the output consistent
                synonymous_dict[word] = synonymous_list
            processed_words.add(word)
    
    return synonymous_dict

In [ ]:
synonymous_dict = find_synonymous_in_list(final_list_of_criteria)
final_criteria = {}
for criteria in synonymous_dict:
    final_criteria[criteria] = final_summarized_criteria[conversion_dict_final[criteria]]

In [ ]:
with open(final_summarized_criteria_output_path, "w") as outfile:
    json.dump(final_criteria, outfile, indent = 4)

# Checkpoint 3: Run QuantifierAgent (All Criteria At Once Version)

In [ ]:
# load the criteria
with open(final_summarized_criteria_output_path) as crit_file:
    criteria = json.load(crit_file)

In [ ]:
def run_quantifier_agent_all_criteria_at_once(num_seeds):
    quantifier_message_base = """You are a helpful assistant. You quantify the output of different tasks based on the given criteria.
    The criterion is given in a dictionary format where each key is a distinct criteria.
    The value of each key is a dictionary as follows {"description": criteria description , "accepted_values": possible accepted inputs for this key}
    You are going to quantify each of the crieria for a given task based on the task decription.
    Return a dictionary where the keys are the criteria and the values are the assessed performance based on accepted values for each criteria.
    Return only the dictionary as a json format string and nothing else."""

    for seed in range(num_seeds):
        outcome = {}
        quantifier = autogen.AssistantAgent(
            name = "quantifier",
            llm_config = {"config_list": config_list,"cache_seed":seed},
            system_message = quantifier_message_base)

        for gameid in tqdm(original_test_cases):
            actual_label, test_case = original_test_cases[gameid]
    
            result = {"actual_success": actual_label, "estimated_performance": {}}

            quantifier_user = autogen.UserProxyAgent(
                name = "quantifier_user",
                max_consecutive_auto_reply = 0,  # terminate without auto-reply
                human_input_mode = "NEVER",
            )
            cq_results = quantifier_user.initiate_chat(quantifier, message = sys_msg + \
                                            "Evaluation dictionary: " + str(criteria) + "\n" + \
                                            "actual test case to evaluate: " + test_case)
            quantified_result = quantifier_user.last_message()["content"]
            
            quantified_result = re.findall(re.compile('{[\s\S]+}'), quantified_result)[0]
            try:
                result["estimated_performance"] = json.loads(quantified_result)
            except:
                result["estimated_performance"] = {}
                print(sys.exc_info()[0])
                
            outcome[gameid] = json.dumps(result)
                
                            
        # store the evaluated problems
        with open(quantifier_agent_all_at_once_output_path_prefix + str(seed) + ".json","w") as file:
            json.dump(outcome,file,indent=2) 

In [ ]:
run_quantifier_agent_all_criteria_at_once(num_quantifier_seeds)

# Checkpoint 4: Run QuantifierAgent (One Criteria At a Time Version)

In [ ]:
def run_quantifier_agent_one_criterion_a_time(num_seeds):
    quantifier_message_base = """You are a helpful assistant. You quantify the output of different tasks based on the given criteria.
        You will be given a criterion a dictionary as follows {"description": criterion description , "accepted_values": possible accepted inputs for this key}.
        You are going to evaluate the test case against the given criterion for the given task.
        Return the assessed performance based on accepted values for each criteria, which must be one of the values provided in the accepted_values list.
        Return only the assessed performance and nothing else"""

    for seed in range(num_seeds):
        outcome = {}
        quantifier = autogen.AssistantAgent(
            name = "quantifier",
            llm_config = {"config_list": config_list,"cache_seed":seed},
            system_message = quantifier_message_base)

        for gameid in tqdm(original_test_cases):
            actual_label, test_case = original_test_cases[gameid]
    
            result = {"actual_success": actual_label, "estimated_performance": {}}

            for criterion in criteria:
                quantifier_user = autogen.UserProxyAgent(
                    name = "quantifier_user",
                    max_consecutive_auto_reply = 0,  # terminate without auto-reply
                    human_input_mode = "NEVER",
                )
                cq_results = quantifier_user.initiate_chat(quantifier, message = sys_msg + \
                                                "Evaluation criterion: " + criterion + "\n" + \
                                                "Evaluation dictionary: " + str(criteria[criterion]) + "\n" + \
                                                "actual test case to evaluate: " + test_case)
                quantified_result = quantifier_user.last_message()["content"]

                quantified_result = re.findall(re.compile('{[\s\S]+}'), quantified_result)[0]
                try:
                    result["estimated_performance"][criterion] = quantified_result
                except:
                    result["estimated_performance"][criterion] = {}
                    print(sys.exc_info()[0])
                        
            outcome[gameid] = json.dumps(result)
                            
        # store the evaluated problems
        with open(quantifier_agent_one_at_a_time_output_path_prefix + str(seed) + ".json","w") as file:
            json.dump(outcome,file,indent=2) 

In [ ]:
run_quantifier_agent_one_criterion_a_time(num_quantifier_seeds)

# Checkpoint 5: Plot QuantifierAgent Standard Deviation

In [ ]:
# convert accepted value to score
level2score = defaultdict(lambda: defaultdict(int))
for criterion in criteria:
     score = 0
     for v in criteria[criterion]["accepted_values"]:
        level2score[criterion][v] = score
        score += 1
print(level2score)

In [ ]:
def get_stdevs_from_quantified_output(num_seeds, path_prefix):
    results = defaultdict(lambda: defaultdict(list)) # result[criterion][game] gives a list of num_seeds scores for that game 
    key_errors = defaultdict(int)
    
    for seed in range(num_seeds):
        # Load outcome data for the current seed
        with open(path_prefix + str(seed) + ".json", 'r') as file:
            outcome = json.load(file)
    
            for game in outcome:
                gameid = game.strip(".json")
                quantified_result = json.loads(outcome[game])['estimated_performance']

                for criterion in quantified_result:
                    quantified_result[criterion] = str(quantified_result[criterion])
                    cleaned_level = quantified_result[criterion].strip("'")
                    results[criterion][gameid].append(level2score[criterion][cleaned_level])
        
    stdevs = defaultdict(lambda: defaultdict(float))
    
    for criterion in results:
        for game in results[criterion]:
            stdevs[criterion][game] = statistics.stdev(results[criterion][game])
        
    stdevs_list = {}
    for criterion in stdevs:
        stdevs_list[criterion] = []
        for game in stdevs[criterion]:
            stdevs_list[criterion].append(stdevs[criterion][game])

    mean_stdevs = {}
    for criterion in stdevs_list:
        mean_stdevs[criterion] = statistics.mean(stdevs_list[criterion])

    return (stdevs_list, mean_stdevs, key_errors, results)

In [ ]:
exp1_stdevs_list, exp1_mean_stdevs, exp1_key_errors, exp1_results = \
get_stdevs_from_quantified_output(num_quantifier_seeds, quantifier_agent_all_at_once_output_path_prefix)
exp2_stdevs_list, exp2_mean_stdevs, exp2_key_errors, exp2_results = \
get_stdevs_from_quantified_output(num_quantifier_seeds, quantifier_agent_one_at_a_time_output_path_prefix)

In [ ]:
for key in original_test_cases:
    print(key)
    print(original_test_cases[key])
    for cr in exp1_results:
        print(cr, exp1_results[cr][str(key)])
    break

In [ ]:
def plot_quantifier_stdev():
    plt.figure(figsize=(12, 8))
    bar_width = 0.2
    index = np.arange(len(criteria))
    
    criteria_names = list(criteria.keys())
    
    exp1_data = []
    exp2_data = []
    for criterion in criteria_names:
        exp1_data.append(exp1_mean_stdevs[criterion])
        exp2_data.append(exp2_mean_stdevs[criterion])
    
    plt.bar(index , exp1_data, bar_width, label=f"all in one", color="blue", capsize=5)
    plt.bar(index + bar_width, exp2_data, bar_width, label=f"one criterion per prompt", color="orange", capsize=5)
    
    plt.xlabel("Criteria", fontsize=16)
    plt.ylabel("Mean Standard Deviation", fontsize=16)
    plt.xticks(index + bar_width / 2, criteria_names, rotation=90, fontsize=14)
    plt.legend(loc="upper center", fontsize=14, bbox_to_anchor=(0.5, 1), ncol=3)  # Adjust legend placement and ncol
    plt.tight_layout()  # Adjust subplot parameters to fit the labels
    plt.savefig(quantifier_stdev_plot_output_path)
    plt.show()

In [ ]:
plot_quantifier_stdev()

# Checkpoint 6: Generate Adversarial Samples

In [ ]:
adversarial_test_cases = {}
for gameid in original_test_cases:
    actual_label, test_case = original_test_cases[gameid]

    adversarial_test_case = create_adversarial_samples(test_case)
    adversarial_test_cases[gameid] = adversarial_test_case
    
with open(adversarial_examples_output_path,"w") as file:
    json.dump(adversarial_test_cases,file,indent=2) 

# Checkpoint 7: Run QuantifierAgent on Adversarial Samples

In [ ]:
def run_quantifier_on_adversarial_samples():
    quantifier_message_base = """You are a helpful assistant. You quantify the output of different tasks based on the given criteria.
        You will be given a criterion a dictionary as follows {"description": criterion description , "accepted_values": possible accepted inputs for this key}.
        You are going to evaluate the test case against the given criterion for the given task.
        Return the assessed performance based on accepted values for each criteria, which must be one of the values provided in the accepted_values list.
        Return only the assessed performance and nothing else"""

    quantifier = autogen.AssistantAgent(
        name = "quantifier",
        llm_config = {"config_list": config_list},
        system_message = quantifier_message_base)

    outcome = {}
    
    for gameid in original_test_cases:
        actual_label, test_case = original_test_cases[gameid]
        adversarial_test_case = adversarial_test_cases[gameid]

        result = {"actual_success": actual_label, "estimated_performance": {}}

        for criterion in criteria:
            quantifier_user = autogen.UserProxyAgent(
                name = "quantifier_user",
                max_consecutive_auto_reply = 0,  # terminate without auto-reply
                human_input_mode = "NEVER",
            )
            cq_results = quantifier_user.initiate_chat(quantifier, message = sys_msg + \
                                            "Evaluation criterion: " + criterion + "\n" + \
                                            "Evaluation dictionary: " + str(criteria[criterion]) + "\n" + \
                                            "actual test case to evaluate: " + test_case)
            quantified_result = quantifier_user.last_message()["content"]

            original_result = quantified_result

            quantifier_user = autogen.UserProxyAgent(
                name = "quantifier_user",
                max_consecutive_auto_reply = 0,  # terminate without auto-reply
                human_input_mode = "NEVER",
            )
            cq_results = quantifier_user.initiate_chat(quantifier, message = sys_msg + \
                                            "Evaluation criterion: " + criterion + "\n" + \
                                            "Evaluation dictionary: " + str(criteria[criterion]) + "\n" + \
                                            "actual test case to evaluate: " + adversarial_test_case)
            quantified_result = quantifier_user.last_message()["content"]

            adversarial_result = quantified_result

            result['estimated_performance'][criterion] = [original_result, adversarial_result]        
                
        outcome[gameid] = json.dumps(result)
                        
    # store the evaluated problems
    with open(adversarial_testing_evaluated_results_output_path,"w") as file:
        json.dump(outcome,file,indent=2) 

In [ ]:
run_quantifier_on_adversarial_samples()

# Checkpoint 8: Plot QuantifierAgent Adversarial Testing Performance

In [ ]:
# load data
with open(adversarial_testing_evaluated_results_output_path) as file:
    original_outcome = json.load(file)

original_adversarial_exp_scores = defaultdict(lambda: defaultdict(list))

for game in original_outcome:
    evaluated_results = json.loads(original_outcome[game])['estimated_performance']
    for criterion in evaluated_results:
        original_level, adversarial_level = evaluated_results[criterion]
        
        original_score = level2score[criterion][original_level.strip('"').strip("'")]
        adversarial_score = level2score[criterion][adversarial_level.strip('"').strip("'")]
        
        original_adversarial_exp_scores[criterion]['original'].append(original_score)
        original_adversarial_exp_scores[criterion]['adversarial'].append(adversarial_score)

original_averages = defaultdict(lambda: defaultdict(float))

for criterion in original_adversarial_exp_scores:
    original_averages[criterion]["original"] =  statistics.mean(original_adversarial_exp_scores[criterion]["original"])
    original_averages[criterion]["adversarial"] =  statistics.mean(original_adversarial_exp_scores[criterion]["adversarial"])

In [ ]:
def plot_quantifier_adversarial_testing():
    plt.figure(figsize=(12, 8))
    bar_width = 0.2
    index = np.arange(len(criteria))
    
    criteria_names = list(criteria.keys())
    
    exp1_data = []
    exp2_data = []
    
    for criterion in criteria_names:
        exp1_data.append(original_averages[criterion]['original'])
        exp2_data.append(original_averages[criterion]['adversarial'])
    
    plt.bar(index , exp1_data, bar_width, label=f"original sample", color="darkblue", capsize=5)
    plt.bar(index + bar_width, exp2_data, bar_width, label=f"adversarial sample", color="lightblue", capsize=5)
    
    plt.xlabel("Criteria", fontsize=16)
    plt.ylabel("Average Quantified Score", fontsize=16)
    plt.xticks(index + bar_width / 2, criteria_names, rotation=90, fontsize=14)
    plt.legend(loc="upper center", fontsize=14, bbox_to_anchor=(0.5, 1), ncol=3)  # Adjust legend placement and ncol
    plt.tight_layout()  # Adjust subplot parameters to fit the labels
    plt.savefig(quantifier_adversarial_testing_plot_output_path)
    plt.show()

In [ ]:
plot_quantifier_adversarial_testing()